# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lunoxas/FlyRank_AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

parquet_files = [f for f in files if f.endswith(".parquet")]

print(f"Found {len(parquet_files)} Parquet files.")

for file in parquet_files:
    print(file)

Found 22 Parquet files.
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month

In [13]:
# 1. Check row count and date coverage
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT month) AS distinct_months
    FROM read_parquet('{march_file}')
""").show()


# 2. Check whether the expected grain is unique:
# one row per report_date + client_hash_id + content_hash_id
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            report_date,
            client_hash_id,
            content_hash_id
        )) AS distinct_grain_combinations
    FROM read_parquet('{march_file}')
""").show()


# 3. Find duplicate grain combinations, if any
con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM read_parquet('{march_file}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").show()


# 4. Check for missing values in the grain columns
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE report_date IS NULL) AS null_report_dates,
        COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS null_client_ids,
        COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS null_content_ids
    FROM read_parquet('{march_file}')
""").show()

┌────────────┬────────────┬────────────┬─────────────────┐
│ total_rows │ first_date │ last_date  │ distinct_months │
│   int64    │    date    │    date    │      int64      │
├────────────┼────────────┼────────────┼─────────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │               1 │
└────────────┴────────────┴────────────┴─────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────────────┐
│ total_rows │ distinct_grain_combinations │
│   int64    │            int64            │
├────────────┼─────────────────────────────┤
│    9841378 │                     9841378 │
└────────────┴─────────────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬─────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ duplicate_count │
│    date     │    varchar     │     varchar     │      int64      │
├─────────────┴────────────────┴─────────────────┴─────────────────┤
│                              0 rows                              │
└──────────────────────────────────────────────────────────────────┘

┌───────────────────┬─────────────────┬──────────────────┐
│ null_report_dates │ null_client_ids │ null_content_ids │
│       int64       │      int64      │      int64       │
├───────────────────┼─────────────────┼──────────────────┤
│                 0 │               0 │                0 │
└───────────────────┴─────────────────┴──────────────────┘



## 1. Unit of analysis + time window

This analysis uses the FlyRank warehouse dataset hosted on Hugging Face.

The selected table is `fact_content_daily_performance`, which contains daily
performance records for pseudonymized clients and content items.

For development and initial validation, I selected the March 2026 partition:
`month=2026-03`.

March 2026 is used as a development partition rather than the final-month
sample. The final month, June 2026, is reserved as a sealed test period
to reduce the risk of developing label logic on the outcome window.


In [6]:
%%sql
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS row_count
FROM fact_content_daily_performance
GROUP BY
    report_date,
    client_id,
    content_id
HAVING COUNT(*) > 1
LIMIT 5;

UsageError: Cell magic `%%sql` not found.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Missing values and data availability

The March 2026 partition contains 9,841,378 rows.

The selected GSC metrics, `gsc_impressions` and `gsc_clicks`, contain
no null values.

The selected GA4 and AI metrics, `ga4_pageviews`, `ga4_sessions`, and
`sessions_ai`, each contain 3,018,741 null values, approximately
30.67% of the partition.

The missingness is associated with the GA4 availability fields:

- 3,018,741 rows have `client_has_ga4 = false` and
  `ga4_data_available = NULL`. The selected metrics are null.
- 6,408,671 rows have `client_has_ga4 = true` and
  `ga4_data_available = false`. The selected metrics are zero-filled.
- 413,966 rows have `client_has_ga4 = true` and
  `ga4_data_available = true`. The selected metrics are populated;
  some observed values are zero.

This demonstrates that missing values, zero-filled values, and observed
zero activity must not be treated as interchangeable.

The availability flags should be retained and considered during
feature engineering. Missing metrics should not be blindly replaced
with zero without accounting for data availability.

In [16]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS null_gsc_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS null_gsc_clicks,

        COUNT(*) FILTER (
            WHERE ga4_pageviews IS NULL
        ) AS null_ga4_pageviews,

        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS null_ga4_sessions,

        COUNT(*) FILTER (
            WHERE sessions_ai IS NULL
        ) AS null_sessions_ai

    FROM read_parquet('{march_file}')
""").show()

┌────────────┬──────────────────────┬─────────────────┬────────────────────┬───────────────────┬──────────────────┐
│ total_rows │ null_gsc_impressions │ null_gsc_clicks │ null_ga4_pageviews │ null_ga4_sessions │ null_sessions_ai │
│   int64    │        int64         │      int64      │       int64        │       int64       │      int64       │
├────────────┼──────────────────────┼─────────────────┼────────────────────┼───────────────────┼──────────────────┤
│    9841378 │                    0 │               0 │            3018741 │           3018741 │          3018741 │
└────────────┴──────────────────────┴─────────────────┴────────────────────┴───────────────────┴──────────────────┘



In [18]:
con.sql(f"""
    SELECT
        client_has_ga4,
        ga4_data_available,
        COUNT(*) AS row_count,

        COUNT(*) FILTER (
            WHERE ga4_pageviews IS NULL
        ) AS null_ga4_pageviews,

        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS null_ga4_sessions,

        COUNT(*) FILTER (
            WHERE sessions_ai IS NULL
        ) AS null_sessions_ai,

        COUNT(*) FILTER (
            WHERE ga4_pageviews = 0
        ) AS zero_ga4_pageviews,

        COUNT(*) FILTER (
            WHERE ga4_sessions = 0
        ) AS zero_ga4_sessions

    FROM read_parquet('{march_file}')
    GROUP BY client_has_ga4, ga4_data_available
    ORDER BY client_has_ga4, ga4_data_available
""").show()

┌────────────────┬────────────────────┬───────────┬────────────────────┬───────────────────┬──────────────────┬────────────────────┬───────────────────┐
│ client_has_ga4 │ ga4_data_available │ row_count │ null_ga4_pageviews │ null_ga4_sessions │ null_sessions_ai │ zero_ga4_pageviews │ zero_ga4_sessions │
│    boolean     │      boolean       │   int64   │       int64        │       int64       │      int64       │       int64        │       int64       │
├────────────────┼────────────────────┼───────────┼────────────────────┼───────────────────┼──────────────────┼────────────────────┼───────────────────┤
│ false          │ NULL               │   3018741 │            3018741 │           3018741 │          3018741 │                  0 │                 0 │
│ true           │ false              │   6408671 │                  0 │                 0 │                0 │            6408671 │           6408671 │
│ true           │ true               │    413966 │                  0 │          

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limitations

The March 2026 partition provides a large number of daily performance
records, but several limitations affect how the data can be interpreted
and used for feature engineering.

### 1. Uneven historical coverage

The warehouse is an unbalanced panel. Clients have different amounts
of historical data, so a single calendar window may not represent the
same amount of history for every client.

Before defining feature or label windows, client-specific history
coverage should be checked using `dim_clients`, particularly
`gsc_data_start` and `ga4_data_start`.

### 2. Data availability and zero-filled values

The availability flags must be considered when interpreting metrics.

In the March 2026 partition, 3,018,741 rows have no GA4 configuration
and contain NULL values for the selected GA4 metrics. A further
6,408,671 rows have GA4 configured but marked unavailable; the selected
GA4 metrics are zero-filled in these rows.

Therefore, zero values cannot automatically be interpreted as
observed zero activity. Missing values, unavailable measurements,
and measured zeros must be distinguished.

### 3. Availability flags can be NULL

The availability fields are not always simple TRUE/FALSE indicators.
For example, `ga4_data_available` is NULL for rows where
`client_has_ga4 = false`.

The analysis must preserve and interpret these states rather than
assuming NULL means FALSE.

### 4. Limited usable history for some clients

The dataset guidance notes that approximately one-third of clients
have little or no usable search or analytics history. This may reduce
the number of clients or observations suitable for particular analyses.

### 5. Development-window limitation

This analysis currently validates the March 2026 partition. The results
confirm the grain and data patterns for that partition, but do not by
themselves prove that every monthly partition has identical quality
or availability patterns.

The final month, June 2026, should remain a sealed test period when
developing past-to-future labels.

### 6. Implications for feature engineering

Features must use information available before the prediction point.
Fields derived from future outcomes or overlapping outcome windows
must be excluded to avoid data leakage.

Availability flags should be retained, and missing values should not
be blindly replaced with zero.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.